# Spending Pattern Analysis with K-Means

## Load Packages

In [ ]:
import pandas as pd
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import silhouette_score, davies_bouldin_score

## Load Dataset

In [ ]:
df = pd.read_csv('spending_l9_dataset.csv')
print(df.head())

## Prepare Features

In [ ]:
FEATURES = ['Income_$', 'SpendingScore']
X = df[FEATURES].copy()

# Fill missing values with median
for col in FEATURES:
    if X[col].isna().any():
        X[col] = X[col].fillna(X[col].median())

print(X.head())

## Scale Features

In [ ]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
print(f'Scaled shape: {X_scaled.shape}')

## Elbow Method (SSE)

In [ ]:
print('=== ELBOW METHOD (SSE per k) ===')
for k in range(1, 11):
    km = KMeans(n_clusters=k, n_init='auto', random_state=42)
    km.fit(X_scaled)
    print(f'k={k} → SSE={km.inertia_:.2f}')

## Train K-Means Model

In [ ]:
K = 3
kmeans = KMeans(n_clusters=K, n_init='auto', random_state=42)
labels = kmeans.fit_predict(X_scaled)

df['Cluster'] = labels.astype(int)
print(f'\n=== SAMPLE WITH {K} CLUSTERS ===')
print(df.head())

## Evaluate Clustering

In [ ]:
sil = silhouette_score(X_scaled, labels)
dbi = davies_bouldin_score(X_scaled, labels)

print('\n=== METRICS ===')
print(f'Silhouette Score : {sil:.3f}')
print(f'Davies–Bouldin   : {dbi:.3f}')

## Cluster Centers (Original Units)

In [ ]:
centers_scaled = kmeans.cluster_centers_
centers_original = scaler.inverse_transform(centers_scaled)

centers_df = pd.DataFrame(centers_original, columns=FEATURES)
centers_df.index.name = 'Cluster'

print('\n=== CLUSTER CENTERS (Original Units) ===')
print(centers_df.round(2))

## Sanity Check

In [ ]:
sample_idx = [0, 5, 10]
sanity = df.loc[sample_idx, FEATURES + ['Cluster']]

print('\n=== SANITY CHECK (3 Customers) ===')
print(sanity)

## Save Output

In [ ]:
df.to_csv('spending_labeled_clusters.csv', index=False)
print('\nSaved: spending_labeled_clusters.csv')